In [0]:
/* ======================================================================
   STEP 0 — EXPLORE RAW DATA
====================================================================== */

-- Explore the raw annual operator statistics
SELECT * 
FROM gshen_catalog.enbridge_sr_workshop.annual_hazardous_liquid_2024_updated
LIMIT 50;

-- Explore the raw hazardous liquid accident data
SELECT *
FROM gshen_catalog.enbridge_sr_workshop.accident_hazardous_liquid_jan_2010_present
LIMIT 50;

/* ======================================================================
   STEP 1 — BUILD THE CLEAN ACCIDENT TABLE
   Purpose:
     - Convert timestamps
     - Standardize naming
     - Extract severity & context metrics
     - Keep only fields needed for benchmarking, AI, or BI
====================================================================== */

CREATE OR REPLACE MATERIALIZED VIEW gshen_catalog.enbridge_sr_workshop.accidents AS
SELECT
  /* Identifiers */
  OPERATOR_ID,
  NAME AS operator_name,
  REPORT_NUMBER,

  /* Time */
  to_timestamp(LOCAL_DATETIME) AS incident_ts,
  year(to_timestamp(LOCAL_DATETIME)) AS incident_year,

  /* Severity metrics */
  UNINTENTIONAL_RELEASE_BBLS AS barrels_released,
  RECOVERED_BBLS,
  IGNITE_IND,
  EXPLODE_IND,
  NUM_EMP_FATALITIES + NUM_GP_FATALITIES AS fatalities,
  NUM_EMP_INJURIES + NUM_GP_INJURIES AS injuries,

  /* Context fields */
  COMMODITY_RELEASED_TYPE,
  COMMODITY_SUBTYPE,
  SYSTEM_PART_INVOLVED,
  ACCIDENT_DETAILS,

  /* Geography */
  ONSHORE_STATE_ABBREVIATION AS state,
  LOCATION_LATITUDE,
  LOCATION_LONGITUDE,

  /* Free Text */
  NARRATIVE

FROM 
  gshen_catalog.enbridge_sr_workshop.accident_hazardous_liquid_jan_2010_present;

-- Inspect the clean accident table
SELECT * FROM gshen_catalog.enbridge_sr_workshop.accidents LIMIT 50;

/* ======================================================================
   STEP 2 — BUILD THE CLEAN ANNUAL OPERATOR TABLE
   Purpose:
     - Bring in annual operator-level pipeline mileage
     - Required for normalized benchmarking (incidents / 1000 miles)
     - Keep only relevant high-value fields
====================================================================== */

CREATE OR REPLACE MATERIALIZED VIEW gshen_catalog.enbridge_sr_workshop.annual AS
SELECT
    OPERATOR_ID,
    REPORT_YEAR,
    PARTA2NAMEOFCOMP AS operator_name,

    /* Total system mileage (normalization denominator) */
    PARTDTOTALMILES AS total_miles,

    /* High Consequence Area exposure metrics */
    PARTBHCAONSHORE AS hca_onshore_miles,
    PARTBHCAOFFSHORE AS hca_offshore_miles

FROM 
    gshen_catalog.enbridge_sr_workshop.annual_hazardous_liquid_2024_updated;

-- Inspect clean annual table
SELECT * FROM gshen_catalog.enbridge_sr_workshop.annual LIMIT 50;

/* ======================================================================
   STEP 3 — BUILD THE INCIDENTS_BASE TABLE (FOR BENCHMARKING)
   Purpose:
     - LEFT JOIN accidents to annual stats by operator + year
     - Produce a unified table with all metrics needed for ranking,
       benchmarking, AI analysis, and BI dashboards.
     - Compute incident-level normalized KPIs
====================================================================== */

CREATE OR REPLACE MATERIALIZED VIEW gshen_catalog.enbridge_sr_workshop.incidents_base AS
SELECT
  /* Core identifiers */
  a.OPERATOR_ID,
  a.operator_name,
  a.incident_year,
  a.REPORT_NUMBER,

  /* Free Text */
  a.NARRATIVE,

  /* Geography */
  a.state,
  a.LOCATION_LATITUDE,
  a.LOCATION_LONGITUDE,

  /* Accident context */
  a.COMMODITY_RELEASED_TYPE,
  a.SYSTEM_PART_INVOLVED,
  a.ACCIDENT_DETAILS,

  /* Severity */
  a.barrels_released,
  a.RECOVERED_BBLS,
  a.IGNITE_IND,
  a.EXPLODE_IND,
  a.fatalities,
  a.injuries,

  /* Annual stats (exposure) */
  s.total_miles,
  s.hca_onshore_miles,
  s.hca_offshore_miles,

  /* Normalized KPI: incident contribution per 1,000 miles
     Used for benchmarking Enbridge vs peers */
  CASE
    WHEN s.total_miles > 0 THEN (1.0 / s.total_miles) * 1000
    ELSE NULL
  END AS incident_weight_per_1000_miles,

  /* Normalized KPI: barrels released per mile */
  CASE 
    WHEN s.total_miles > 0 THEN a.barrels_released / s.total_miles
    ELSE NULL
  END AS barrels_per_mile

FROM 
    gshen_catalog.enbridge_sr_workshop.accidents a
LEFT JOIN 
    gshen_catalog.enbridge_sr_workshop.annual s
      ON a.OPERATOR_ID = s.OPERATOR_ID
     AND a.incident_year = s.REPORT_YEAR;

-- Inspect the combined view
SELECT * 
FROM gshen_catalog.enbridge_sr_workshop.incidents_base
LIMIT 50;
